# DRAEM Anomaly Detection: Master Reproduction Notebook

Reproduces the DRAEM pipeline for the ADL 2025-2026 challenge.

Pipeline (single script `scripts/train_evaluate_draem_v2.py`):
1. **Training** — reconstructive subnetwork (encoder/decoder, no skip) + discriminative subnetwork (U-Net) on synthetic anomalies. Each training step blends a DTD texture patch into a clean image inside a random fractal-noise mask, then trains the reconstructive net to recover the clean image (MSE) and the discriminative net to predict the mask (focal loss). Plus an auxiliary margin loss on the few labeled real anomaly masks.
2. **Local evaluation + PDF** — anomaly probability maps on the labeled training anomalies for manual inspection (NOTE: training-set overlap, optimistic).
3. **Submission** — per-class percentile-clipped + multi-view sample gate → q8rle CSV/ZIP.

**Texture source:** the Describable Textures Dataset (DTD) is auto-downloaded on first run via `torchvision.datasets.DTD` to `data/dtd/` (~600 MB).

**Note:** Designed to run on Google Colab or locally with the correct environment.

## 1. Setup and Environment

In [1]:
!nvidia-smi

Sun May 17 21:38:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.142                Driver Version: 580.142        CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4070 ...    Off |   00000000:01:00.0 Off |                  N/A |
| 30%   40C    P3             21W /  220W |     100MiB /  12282MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# If running on Colab, uncomment and run this:
# !pip install timm opencv-python pandas tqdm matplotlib scipy torchvision

In [3]:
import os
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")
print(f"Current working directory: {os.getcwd()}")

Current working directory: /home/max/Documents/ghi8/adl/anomaly-detection


### DTD texture dataset (auto-download)

On first run, `torchvision.datasets.DTD` downloads to `./data/dtd/` (~600 MB). After that the loader reuses the cached files. You can pre-trigger the download by running the cell below; otherwise the v2 script triggers it on first batch fetch.

In [4]:
from pathlib import Path
dtd_root = Path("data/dtd")
marker = dtd_root / "dtd" / "images"
if marker.exists():
    print(f"DTD present at {dtd_root}")
else:
    print(f"DTD not present at {dtd_root} — will be auto-downloaded on first script run.")
    print("To pre-download now, uncomment the next two lines:")
    print("# from torchvision.datasets import DTD")
    print("# _ = DTD(root='data/dtd', split='train', download=True)")

DTD not present at data/dtd — will be auto-downloaded on first script run.
To pre-download now, uncomment the next two lines:
# from torchvision.datasets import DTD
# _ = DTD(root='data/dtd', split='train', download=True)


In [5]:
# from torchvision.datasets import DTD
# _ = DTD(root='data/dtd', split='train', download=True)

## 2. Execution Pipeline

The v2 script runs training (with DTD-based synthetic anomalies), local evaluation + PDF generation, and submission encoding in one pass.

Tweakable flags:
- `--epochs` (default 15)
- `--lr` (default 1e-4)
- `--base_channels` (default 64) — width of both U-Nets; lower it (e.g. 32) to fit smaller GPUs
- `--focal_gamma` (default 2.0) — focal-loss focusing parameter
- `--p_aug` (default 0.5) — probability of applying a synthetic anomaly per training image (the rest are clean → mask = 0, teaches the discriminator to abstain)
- `--w_aux` (default 0.05) — weight on the auxiliary supervised mask loss; set to 0 to disable
- `--aux_margin` (default 0.1) — margin for the auxiliary loss
- `--view_gate_alpha` (default 0.5) — sample-level multi-view multiplicative gate strength; set to 0 to disable
- `--classes class_01 class_02 ...` (default: all classes found under `PATH`)

Outputs:
- `artifacts/draem_v2/metrics.csv`
- `artifacts/draem_v2/predictions.pdf`
- `submission/submission_draem_v2.csv`
- `submission/submission_draem_v2.zip`

In [6]:
# Train + local eval + PDF + submission (one pass)
%run scripts/train_evaluate_draem_v2.py --base_channels 32

Processing 8 classes with DRAEM
Using batch_size=32, base_channels=32, amp=True, accum_steps=1

>>> Class: class_01
Training on 2600 good + 25 labeled anomaly samples...


KeyboardInterrupt: 

## 3. Kaggle Submission

Uncomment and run the following cell to submit the results directly to Kaggle.

In [ ]:
# !kaggle competitions submit -c adl-2025-2026-anomaly-detection -f submission/submission_draem_v2.zip -m "DRAEM v2 (synthetic-anomaly recon + discriminator + aux mask + view gate)"